# 2.1 — Register Experiment in RHOAI Model Registry

Notebook 2 already uploaded `models/fraud/1/model.onnx` to S3. This notebook:
1. Evaluates the saved ONNX model on the held-out test set to harvest final metrics.
2. Registers a `ModelVersion` in the RHOAI Model Registry with training hyperparameters and evaluation metrics pointing at the S3 path from notebook 2.

**Prerequisites (Keras path):** Run `1_tf_experiment_train.ipynb` then `2_save_model.ipynb` first.

**Prerequisites (KFTO path):** Run `6_torch_distributed_train.ipynb` first (it uploads to S3 directly).

## Install dependencies

In [ ]:
%%capture
%pip install model-registry onnxruntime --extra-index-url https://pypi.org/simple/

In [ ]:
import os
import pickle
import datetime
from pathlib import Path

import numpy as np
import onnxruntime as rt
from sklearn.metrics import precision_score, recall_score
from model_registry import ModelRegistry

## 2.1a. Evaluate the model and collect experiment metadata

Load the saved ONNX model and the held-out test set (both written by notebook 1) to compute final evaluation metrics.

In [ ]:
MODEL_NAME    = "fraud"
MODEL_VERSION = "1"
LOCAL_ONNX    = Path(f"models/{MODEL_NAME}/{MODEL_VERSION}/model.onnx")

# Training hyperparameters — match what 1_experiment_train.ipynb uses
TRAIN_PARAMS = {
    "epochs":        2,
    "optimizer":     "adam",
    "loss":          "binary_crossentropy",
    "hidden_layers": 3,
    "hidden_units":  32,
    "dropout":       0.2,
    "batch_norm":    True,
    "threshold":     0.95,
    "features":      "distance_from_last_transaction,ratio_to_median_purchase_price,used_chip,used_pin_number,online_order",
}

if not LOCAL_ONNX.exists():
    raise FileNotFoundError(
        f"{LOCAL_ONNX} not found. Run 1_experiment_train.ipynb first."
    )

# Load saved test data
with open('artifact/test_data.pkl', 'rb') as f:
    X_test, y_test = pickle.load(f)

# Run inference with the saved ONNX model
sess = rt.InferenceSession(
    str(LOCAL_ONNX),
    providers=rt.get_available_providers(),
)
input_name  = sess.get_inputs()[0].name
output_name = sess.get_outputs()[0].name

y_prob = np.squeeze(sess.run([output_name], {input_name: X_test.astype(np.float32)})[0])
y_pred = (y_prob > TRAIN_PARAMS["threshold"]).astype(int)
y_true = y_test.squeeze()

accuracy  = float(np.equal(y_pred, y_true).mean())
precision = float(precision_score(y_true, y_pred))
recall    = float(recall_score(y_true, y_pred))

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")

## 2.1b. Register in RHOAI Model Registry

`MODEL_REGISTRY_URL` is the route of the Model Registry service on your cluster. Obtain it with:

```bash
oc get route -n rhoai-model-registries modelregistry-sample -o jsonpath='{.spec.host}'
```

In [ ]:
bucket_name = os.environ.get('AWS_S3_BUCKET', 'models')
S3_PREFIX   = f"models/{MODEL_NAME}"
s3_uri      = f"s3://{bucket_name}/{S3_PREFIX}/"

def _fmt(v, digits=4):
    try:
        return f"{float(v):.{digits}f}"
    except (TypeError, ValueError):
        return str(v)

# RHOAI dashboard convention:
#   empty-string value  → rendered as a Label chip
#   non-empty string    → rendered as a Property
metadata = {
    # Labels
    "fraud-detection":       "",
    "binary-classification": "",
    "onnx-export":           "",

    # Provenance
    "registered_at": datetime.datetime.now(datetime.UTC).isoformat(timespec="seconds"),

    # Training hyperparameters
    "param.epochs":        _fmt(TRAIN_PARAMS["epochs"],        digits=0),
    "param.optimizer":     TRAIN_PARAMS["optimizer"],
    "param.loss":          TRAIN_PARAMS["loss"],
    "param.hidden_layers": _fmt(TRAIN_PARAMS["hidden_layers"], digits=0),
    "param.hidden_units":  _fmt(TRAIN_PARAMS["hidden_units"],  digits=0),
    "param.dropout":       _fmt(TRAIN_PARAMS["dropout"],       digits=2),
    "param.batch_norm":    str(TRAIN_PARAMS["batch_norm"]),
    "param.threshold":     _fmt(TRAIN_PARAMS["threshold"],     digits=2),
    "param.features":      TRAIN_PARAMS["features"],

    # Evaluation metrics
    "metric.accuracy":     _fmt(accuracy),
    "metric.precision":    _fmt(precision),
    "metric.recall":       _fmt(recall),
}

version_description = (
    f"Fraud detection MLP (3× Dense-32 + BatchNorm + Dropout), exported to ONNX. "
    f"Input: 5 transaction features. "
    f"accuracy={_fmt(accuracy)} precision={_fmt(precision)} recall={_fmt(recall)} "
    f"over {TRAIN_PARAMS['epochs']} epochs."
)

In [ ]:
DATA_CONNECTION = os.environ.get("DATA_CONNECTION", "aws-connection-storage")

# scheme+host only — the client appends the port and API path itself
MODEL_REGISTRY_URL = os.environ.get(
    "MODEL_REGISTRY_URL",
    "https://model-registry-rest.apps.cluster-7hb2t.7hb2t.sandbox670.opentlc.com",
)

# The model registry REST API requires an OCP user token (not the workbench SA token).
# Set MODEL_REGISTRY_TOKEN in the workbench environment variables to the output of:
#   oc whoami -t   (run from a terminal where you are logged in as a cluster user)
token = os.environ.get("MODEL_REGISTRY_TOKEN")
if not token:
    raise RuntimeError(
        "MODEL_REGISTRY_TOKEN is not set.\n"
        "Run `oc whoami -t` from your local terminal and set the token as a "
        "workbench environment variable named MODEL_REGISTRY_TOKEN."
    )

registry = ModelRegistry(
    server_address=MODEL_REGISTRY_URL,
    port=int(os.environ.get("MODEL_REGISTRY_PORT", 443)),
    author=os.environ.get("JUPYTER_USER", "workshop"),
    user_token=token,
    is_secure=True,
)

print(f"Connected to Model Registry: {MODEL_REGISTRY_URL}")

In [ ]:
try:
    rm = registry.register_model(
        name=MODEL_NAME,
        version=MODEL_VERSION,
        model_format_name="onnx",
        model_format_version="1",
        storage_key=DATA_CONNECTION,
        storage_path=S3_PREFIX,
        uri=s3_uri,
        version_description=version_description,
        metadata=metadata,
    )
    print(f"Created new registration: {rm.name}  id={rm.id}")
except Exception as e:
    if "already exists" in str(e).lower() or "409" in str(e):
        rm = registry.get_registered_model(MODEL_NAME)
        print(f"Model already registered (id={rm.id}), updating metadata...")
    else:
        raise

# Update the parent RegisteredModel (Overview tab in RHOAI dashboard)
rm.description = (
    "Binary fraud-detection classifier (fully connected neural network, ONNX export). "
    "Input: 5 transaction features. Output: fraud probability (sigmoid). "
    "Trained on a balanced credit-card transaction dataset."
)
rm.owner = os.environ.get("JUPYTER_USER", "workshop")
rm.custom_properties = {
    # Labels (empty string → chip in the dashboard Labels field)
    "fraud-detection":       "",
    "binary-classification": "",
    "workshop-demo":         "",
    # Properties (non-empty)
    "framework":     "keras / tensorflow",
    "architecture":  "MLP (3 hidden layers)",
    "export_format": "onnx",
    "num_features":  "5",
    "num_classes":   "2",
    "classes":       "not-fraud,fraud",
}
registry.update(rm)

print(f"\nRegisteredModel (Overview tab)  id={rm.id}")
print(f"  description: {rm.description}")
print(f"  owner:       {rm.owner}")
for k, v in sorted(rm.custom_properties.items()):
    kind = "label" if v == "" else "prop "
    print(f"  {kind}  {k:22s} = {v!r}")

print(f"\nModelVersion custom properties:")
for k in sorted(metadata):
    kind = "label" if metadata[k] == "" else "prop "
    print(f"  {kind}  {k:26s} = {metadata[k]!r}")